# The same thing, in PyTorch

> Everything you built by hand, in a tenth of the code — plus the one thing the library genuinely gives you, which is not convenience.

Read this chapter at `/learn/10-pytorch/`. Exported from `src/content/chapters/10-pytorch.mdx` — edit there, not here.


**This chapter needs a local kernel.** PyTorch has no WebAssembly build, so the
browser runtime cannot run it. Start one in a second terminal:

```bash
bun run kernel
```

then click the runtime pill in the header and choose **On this machine**. Cells
below are labelled `needs local kernel` so you always know which is which. Full
instructions are on the [Setup](/setup/) page.

You have now written a forward pass, a backward pass and a training loop with
nothing but NumPy. This chapter throws almost all of it away — and the point is
to know exactly what you are throwing away, because that is what you will be
debugging for the rest of your career.

## Tensors are arrays with two extra fields

In [ ]:
# needs PyTorch (this kernel has it; the browser runtime does not)
import torch
import numpy as np

t = torch.arange(6.).reshape(2, 3)
print(t)
print(f"shape {tuple(t.shape)}  dtype {t.dtype}  device {t.device}  grad {t.requires_grad}")

A tensor is a NumPy array plus **a device** (where the
memory lives) and **a place on the autograd tape** (how this value was computed).
Everything else you already know: broadcasting,
axes, `@`, reshape.

In [ ]:
# needs PyTorch (this kernel has it; the browser runtime does not)
a = torch.randn(4, 5)
print("numpy  a.reshape(-1)   ->  torch  a.view(-1)   ", tuple(a.view(-1).shape))
print("numpy  a.sum(axis=0)   ->  torch  a.sum(dim=0) ", tuple(a.sum(dim=0).shape))
print("numpy  np.concatenate  ->  torch  torch.cat    ", tuple(torch.cat([a, a]).shape))
print("shares memory with numpy:", np.shares_memory(a.numpy(), a.numpy()))

`axis` becomes `dim`, `reshape` also exists as `view` (which requires contiguous
memory and is free), and the default float type is **`float32`, not `float64`**.
That last one catches everybody: `torch.tensor([1, 2, 3])` gives you an *integer*
tensor, which then fails somewhere unhelpful.

`.numpy()` and `torch.from_numpy()` share the buffer rather than copying — the
two objects are different views of the same allocation, and mutating one mutates
the other. It is the same aliasing hazard as a NumPy slice, and the same absence
of a compiler to warn you about it.

## Autograd replaces your backward pass

In [ ]:
# needs PyTorch (this kernel has it; the browser runtime does not)
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 5 * x
y.backward()
print(f"y = x^2 + 5x  at x=3  ->  dy/dx = {x.grad.item()}   (analytically 2*3 + 5 = 11)")

Nothing did algebra. PyTorch recorded that a squaring and a multiplication
happened, and `.backward()` walked that record in reverse
applying the local derivatives — exactly the procedure you implemented by hand
yesterday, generated automatically.

In [ ]:
# needs PyTorch (this kernel has it; the browser runtime does not)
torch.manual_seed(0)
X = torch.randn(50, 2)
y = (X[:, 0] * X[:, 1] > 0).float()

W1 = torch.randn(2, 8, requires_grad=True)
b1 = torch.zeros(8, requires_grad=True)
W2 = torch.randn(8, 1, requires_grad=True)
b2 = torch.zeros(1, requires_grad=True)

a1 = torch.relu(X @ W1 + b1)
p  = torch.sigmoid(a1 @ W2 + b2).squeeze()
loss = torch.nn.functional.binary_cross_entropy(p, y)
loss.backward()

# The same expression you derived in step 2 of chapter 9:
manual_gW2 = a1.T @ ((p - y).unsqueeze(1) / len(y))
print("autograd matches the hand derivation:",
      torch.allclose(W2.grad, manual_gW2, atol=1e-6))

That is the whole of chapter 9, verified. `a1.T @ (prediction - target) / n` is
what the library computes too — it just does it for arbitrary graphs, without you
writing it down.

The real gift is not brevity. It is that autograd is correct for *any* graph you
build, including one you invent this afternoon. Your hand-written backward pass
is correct for exactly the architecture you derived it for, and has to be
rederived and re-gradient-checked every time you change a layer.

## `nn.Module`

In [ ]:
# needs PyTorch (this kernel has it; the browser runtime does not)
import torch.nn as nn

class Net(nn.Module):
    def __init__(self, n_in=2, hidden=32, n_out=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_out),
        )
    def forward(self, x):
        return self.net(x)          # logits, not probabilities — see below

model = Net()
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))

`nn.Module` discovers its parameters by walking the
attributes you assign to `self`. That is how `model.parameters()` enumerates
every weight in a network of arbitrary depth without you keeping a list, and how
`model.to(device)` moves all of them at once.

`nn.Linear(a, b)` holds a weight of shape `(b, a)` and a bias of shape `(b,)`,
and computes `x @ W.T + b`. Same arithmetic as yesterday, transposed by
convention.

Notice `forward` returns **logits** — raw scores — not probabilities. That is not
sloppiness, it is required.

`nn.BCEWithLogitsLoss` and `nn.CrossEntropyLoss` apply the sigmoid or softmax
*internally*, using a numerically stable formulation that never computes
`exp(large)`. If you apply the sigmoid yourself and then use `BCELoss`, you get
the same answer on easy data and `nan` on hard data.

The rule: **your model outputs logits; the loss function owns the squashing.**
Apply sigmoid or softmax only at inference, when you actually want a probability.

## The training loop

In [ ]:
# needs PyTorch (this kernel has it; the browser runtime does not)
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

Xn, yn = make_moons(n_samples=2000, noise=0.22, random_state=0)
Xn = (Xn - Xn.mean(0)) / Xn.std(0)
Xtr, Xva, ytr, yva = train_test_split(Xn, yn, test_size=0.25, random_state=0, stratify=yn)

Xtr = torch.tensor(Xtr, dtype=torch.float32); ytr = torch.tensor(ytr, dtype=torch.float32)
Xva = torch.tensor(Xva, dtype=torch.float32); yva = torch.tensor(yva, dtype=torch.float32)

torch.manual_seed(0)
model = Net()
loss_fn = nn.BCEWithLogitsLoss()
opt = torch.optim.AdamW(model.parameters(), lr=1e-2)

for epoch in range(300):
    model.train()
    opt.zero_grad()                       # 1. clear last step's gradients
    logits = model(Xtr).squeeze()         # 2. forward
    loss = loss_fn(logits, ytr)           # 3. score
    loss.backward()                       # 4. backward
    opt.step()                            # 5. update

    if epoch % 100 == 99:
        model.eval()
        with torch.no_grad():
            acc = ((model(Xva).squeeze() > 0).float() == yva).float().mean()
        print(f"epoch {epoch+1:3d}   loss {loss.item():.4f}   valid acc {acc:.3f}")

**Those five lines are the whole of deep learning.** Every training script you
will ever read — a two-layer toy, a vision model, a language model — is this
sequence, plus logging, plus a data loader.

`opt.zero_grad()` is not optional, and forgetting it does not error.

PyTorch *accumulates* into `.grad` rather than overwriting, because that lets you
split one large batch across several backward passes. The consequence is that
without `zero_grad()`, step 50 uses the sum of the first fifty gradients, the
effective learning rate grows without bound, and training diverges with no
message. It is the most common PyTorch bug in existence.

`model.train()` and `model.eval()` toggle a flag that changes the behaviour of
dropout and batch normalisation — dropout is off at evaluation, and batch norm
uses running statistics instead of batch statistics. On this network they do
nothing, and getting into the habit costs you nothing.
`with torch.no_grad()` stops building the tape,
which saves memory and time when you are not going to call `.backward()`.

## Data loading

In [ ]:
# needs PyTorch (this kernel has it; the browser runtime does not)
from torch.utils.data import TensorDataset, DataLoader

train_dl = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)

torch.manual_seed(0)
model = Net()
opt = torch.optim.AdamW(model.parameters(), lr=1e-2)

for epoch in range(20):
    model.train()
    for xb, yb in train_dl:                      # <- the only change
        opt.zero_grad()
        loss_fn(model(xb).squeeze(), yb).backward()
        opt.step()

model.eval()
with torch.no_grad():
    acc = ((model(Xva).squeeze() > 0).float() == yva).float().mean()
print(f"mini-batch training, valid acc {acc:.3f}")

`Dataset` answers "how many, and give me item *i*".
`DataLoader` turns that into shuffled, batched, optionally parallel-prefetched
tensors. The separation is the good idea: `Dataset` knows your file format and
nothing about batching; `DataLoader` knows about batching and nothing about your
files.

## Devices, and Apple Silicon

In [ ]:
# needs PyTorch (this kernel has it; the browser runtime does not)
def best_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")          # Apple Silicon GPU
    return torch.device("cpu")

dev = best_device()
print("using:", dev)

m = Net().to(dev)
x = torch.randn(256, 2, device=dev)
print("forward on device:", tuple(m(x).shape), m(x).device)

Two tensors on different devices cannot be combined, and the error says so
plainly. The idiom is to choose the device once at the top and route everything
through it.

**MPS** is Apple's Metal backend, and on an Apple Silicon Mac it is a real
speedup over CPU for anything convolutional. Two caveats worth knowing. Some
operations are still unimplemented and fall back to CPU — set
`PYTORCH_ENABLE_MPS_FALLBACK=1` so that happens silently rather than raising. And
`float64` is not supported at all on MPS, which is one more reason everything in
this world is `float32`.

For a network this small, MPS is *slower* than CPU: kernel-launch overhead
dominates when there is barely any arithmetic to do. Device choice is worth
measuring, not assuming.

## What the library actually gave you

In [ ]:
# needs PyTorch (this kernel has it; the browser runtime does not)
print("NumPy version (chapter 9): ~24 lines of init + forward + backward + loop")
print("PyTorch version          : ~11 lines")
print()
for item in [
    "autograd that is correct for ANY graph, not just the one you derived",
    "GPU/MPS execution with a one-word change",
    "fused, tested, numerically-stable loss functions",
    "optimisers you would otherwise reimplement (AdamW, schedulers)",
    "a data pipeline with parallel prefetch",
    "an ecosystem of pretrained weights",
]:
    print(" *", item)

The line count is the least of it. The first item is the one that matters: the
moment you want a residual connection, a custom loss, or an architecture you read
about this morning, the hand-written version needs a fresh derivation and a fresh
gradient check. Autograd does not.

Which is exactly why the two days deriving it by hand were worth spending. When
your loss goes to `nan`, when a gradient is silently zero, when a layer does not
learn — you now know what is happening underneath, and that is the difference
between debugging and guessing.

## The bugs you will hit

In [ ]:
# needs PyTorch (this kernel has it; the browser runtime does not)
# 1. integer tensor where a float was expected
try:
    nn.Linear(3, 1)(torch.tensor([[1, 2, 3]]))
except Exception as e:
    print("1.", type(e).__name__, "->", str(e)[:88])

# 2. calling backward twice on the same graph
z = torch.tensor(2.0, requires_grad=True) ** 2
z.backward()
try:
    z.backward()
except Exception as e:
    print("2.", type(e).__name__, "->", str(e)[:88])

# 3. the silent one: a shape mismatch that broadcasts instead of failing
pred = torch.randn(8, 1)      # model output
targ = torch.randn(8)         # labels
print("3. (pred - targ).shape =", tuple((pred - targ).shape), "  <- should be (8,), got 8x8")

1. **dtype.** `torch.tensor([1, 2, 3])` is `int64`. Write `1.` or pass `dtype=`.
2. **Freed graph.** The tape is discarded after `backward()`. Wanting
   `retain_graph=True` usually means you meant to accumulate the loss and call
   backward once.
3. **Shape.** This is the dangerous one, because it does not raise. `model(x)`
   gives `(n, 1)` and your labels are `(n,)`, so subtraction
   broadcasts into `(n, n)` and your loss is the mean
   of 64 meaningless numbers. `.squeeze()` the output or `.unsqueeze(1)` the
   labels, and print both shapes whenever a loss looks wrong.

Two more with no error message at all: **forgetting `zero_grad()`** (divergence
with no signal) and **forgetting `model.eval()`** (validation scores mysteriously
worse than training, because dropout is still on).

## Exercise

In [ ]:
# needs PyTorch (this kernel has it; the browser runtime does not)
# 1. Rewrite Net without nn.Sequential — plain self.fc1 / self.fc2 attributes
#    and an explicit forward. Confirm the parameter count is unchanged.
#
# 2. Swap AdamW for SGD(lr=0.1), then SGD(lr=0.1, momentum=0.9).
#    Plot all three loss curves on one axis.
#
# 3. Delete opt.zero_grad() and train for a few epochs, printing the gradient
#    norm each time. Watch it grow.
#
# 4. Add nn.Dropout(0.3) between the layers. Train, then evaluate twice —
#    once in model.train() mode and once in model.eval(). Why do they differ?

print("replace me")

In [ ]:
# needs PyTorch (this kernel has it; the browser runtime does not)
# 3. Life without zero_grad
torch.manual_seed(0)
m = Net(); o = torch.optim.SGD(m.parameters(), lr=0.01)
for epoch in range(6):
    # o.zero_grad()   <- deliberately missing
    loss = loss_fn(m(Xtr).squeeze(), ytr)
    loss.backward()
    norm = sum(p.grad.norm().item() ** 2 for p in m.parameters()) ** 0.5
    o.step()
    print(f"epoch {epoch}  loss {loss.item():.4f}  grad norm {norm:8.3f}")

The gradient norm climbs every epoch because each `backward()` adds to what was
already there. By epoch 6 the effective learning rate is several times what you
asked for, and it keeps growing. On a real run this reaches `inf` and then `nan`,
with no error and no clue.

In [ ]:
# needs PyTorch (this kernel has it; the browser runtime does not)
# 4. Dropout behaves differently in train() and eval()
class Dropped(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 32), nn.ReLU(),
                                 nn.Dropout(0.3), nn.Linear(32, 1))
    def forward(self, x): return self.net(x)

torch.manual_seed(0)
d = Dropped(); o = torch.optim.AdamW(d.parameters(), lr=1e-2)
for _ in range(300):
    d.train(); o.zero_grad()
    loss_fn(d(Xtr).squeeze(), ytr).backward(); o.step()

d.train()
with torch.no_grad():
    a = [round((((d(Xva).squeeze() > 0).float() == yva).float().mean()).item(), 4) for _ in range(3)]
d.eval()
with torch.no_grad():
    b = round((((d(Xva).squeeze() > 0).float() == yva).float().mean()).item(), 4)
print("train() mode, three runs:", a, " <- different each time")
print("eval()  mode:            ", b, " <- deterministic")

In `train()` mode dropout randomly zeroes 30% of the hidden units on every
forward pass, so the same input gives a different answer each time. In `eval()`
mode dropout is a no-op and all units contribute.

That is the point of dropout: it forces the network not to depend on any single
unit, because that unit might be missing. It is
[regularisation](/learn/06-generalisation/) by deliberate sabotage, and it is
also why forgetting `model.eval()` produces validation numbers that are
inexplicably worse than they should be.

That is the last chapter where you build things from nothing. From here the
subject is architecture: what structure to impose on the stack, and why.